In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dataset = pd.read_csv("intermediate_data/dataset2.csv")
final_df_non_train = pd.read_csv("intermediate_data/final_non_train2.csv")
app_test_df = pd.read_csv("./data/application_test.csv")

In [ ]:
X = dataset.drop(["SK_ID_CURR", "TARGET"], axis=1)
y = dataset["TARGET"]

In [ ]:
X = X.select_dtypes(include="number")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.to_list()
cat_cols = X_train.columns.difference(num_cols).to_list()
cat_cols

In [ ]:
num_pipeline = Pipeline(
    [
        ("num_impute", SimpleImputer(strategy='mean')),
        ("sc", StandardScaler())
    ]
)
cat_pipeline = Pipeline([
        ("cat_impute", SimpleImputer(strategy='most_frequent')),
        ("ohe", OneHotEncoder())
    ]
)
ct = ColumnTransformer(
    [
        ("num_preprocess", num_pipeline, num_cols),
    ],
    verbose_feature_names_out=False
)

In [ ]:
logreg_pipe = Pipeline([
    ("ct", ct),
    ("model", LogisticRegression(
        solver='saga',
        class_weight='balanced',
        max_iter=500,
        tol=1e-3,
        random_state=42,
    ))
])

In [ ]:
logreg_pipe_nc = Pipeline([
    ("ct", ct),
    ("model", LogisticRegression(
        solver='newton-cholesky',
        C=1,
        class_weight='balanced',
        max_iter=500,
        random_state=42,
        verbose=1
    ))
])

In [ ]:
logreg_pipe_nc.fit(X_train, y_train)

In [ ]:
from scipy.stats import uniform, loguniform

In [ ]:
param_dist = {
    "model__l1_ratio": [0, 0.1, 0.5, 0.9, 1],
    "model__C": [1, 0.1, 0.01],
}

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
random_search = GridSearchCV(
    estimator=logreg_pipe,
    param_grid=param_dist,
    scoring='roc_auc',
    n_jobs=8,
    cv=5,
    verbose=1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
ct.get_feature_names_out()

In [ ]:
coef = logreg_pipe.named_steps["model"].coef_[0]
feature_importance_df = pd.DataFrame({
    "feature": ct.get_feature_names_out(),
    "coef": coef
})

selected_features = feature_importance_df[feature_importance_df["coef"] != 0]

selected_features.sort_values("coef", key=abs, ascending=False)


In [ ]:
selected_features.count()

In [ ]:
selected_features.sort_values("coef", key=abs, ascending=False).head(40)

In [ ]:
selected_cols = selected_features['feature'].values.tolist()
selected_cols.append('SK_ID_CURR')
selected_cols.append('TARGET')
dataset_selected = dataset[selected_cols]
dataset_selected

In [ ]:
cat_col_list = dataset.select_dtypes(exclude='number').columns.to_list()
cat_col_list.append("SK_ID_CURR")
cat_col_list


In [ ]:
dataset_selected = dataset_selected.merge(dataset[cat_col_list], on='SK_ID_CURR', how='left')

In [ ]:
dataset_selected.to_csv("intermediate_data/dataset3.csv")

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
best_estimator = random_search.best_estimator_

In [ ]:
proba = best_estimator.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)

In [ ]:
proba = logreg_pipe_nc.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)